# Lab 5 · Notebook 03 – Self-Training và so sánh

**Foundation (gợi ý HTTT):** tau=0.9, xem số pseudo-label và so sánh supervised.  
**Applied (gợi ý PTPM):** thử tau=0.8 và 0.9, chọn bằng **validation Macro-F1**; sau đó đánh giá trên test đúng **một lần**.

Hai lộ trình có cùng rubric 100 điểm. Pseudo-label có thể sai: **không ép kết quả cao hơn baseline**.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "lab5.py").exists()), None)
assert ROOT, "Hãy chạy Jupyter trong repository FIT4112 Lab 5"
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print("Repo:", ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from src.lab5 import (
    TARGET,FEATURES,train_supervised,train_self_training,metrics,choose_tau
)
TRACK="FOUNDATION"  # hoặc "APPLIED"
folder=ROOT/"data/processed"
fit=pd.read_csv(folder/"masked_fit.csv",parse_dates=["datetime"])
val=pd.read_csv(folder/"validation.csv",parse_dates=["datetime"])
test=pd.read_csv(folder/"heldout_test.csv",parse_dates=["datetime"])
baseline=train_supervised(fit)

if TRACK=="FOUNDATION":
    tau=.90
    model,history=train_self_training(fit,tau=tau)
    note="Foundation: fixed tau=.90, no test-driven selection."
elif TRACK=="APPLIED":
    tau,table,models=choose_tau(fit,val,taus=(.8,.9))
    display(table.round(3))
    model,history=models[tau]
    note="Applied: selected tau by validation Macro-F1; test untouched."
else:
    raise ValueError("TRACK must be FOUNDATION or APPLIED")
print(note,"tau:",tau)
display(history)

### Đánh giá cuối trên cùng test

Ghi lại accuracy, macro-F1 và confusion matrix. **Self-Training có thể kém supervised** do sai số pseudo-label; kết luận phải dựa trên số liệu thực nghiệm.

In [ ]:
base=metrics(baseline,test)
semi=metrics(model,test)
rows=pd.DataFrame([
    {"model":"Supervised baseline","accuracy":base["accuracy"],"macro_f1":base["macro_f1"]},
    {"model":f"Self-Training (tau={tau})","accuracy":semi["accuracy"],"macro_f1":semi["macro_f1"]}
])
display(rows.round(3))
fig,axes=plt.subplots(1,2,figsize=(9,3.5))
for ax,title,model_used in zip(axes,["Supervised","Self-Training"],[baseline,model]):
    ConfusionMatrixDisplay.from_predictions(test[TARGET].astype(int),
                                           model_used.predict(test[FEATURES]),
                                           labels=[0,1],ax=ax,colorbar=False)
    ax.set_title(title)
plt.tight_layout();plt.show()
history.to_csv(folder/"pseudo_label_history.csv",index=False)
rows.to_csv(folder/"model_comparison.csv",index=False)
pd.DataFrame({
    "datetime":test["datetime"],"actual":test[TARGET].astype(int),
    "baseline_pred":baseline.predict(test[FEATURES]),
    "self_training_pred":model.predict(test[FEATURES])
}).to_csv(folder/"test_predictions.csv",index=False)
with (folder/"results.json").open("w",encoding="utf-8") as f:
    json.dump({"track":TRACK,"tau":tau,"supervised":base,"self_training":semi},
              f,ensure_ascii=False,indent=2)
print("Saved results")

### Thảo luận
1. Self-Training có cải thiện Macro-F1 không? Chênh lệch bao nhiêu?
2. Có bao nhiêu pseudo-label được thêm? Vì sao threshold cao có thể làm ít nhãn giả hơn?
3. Sai số pseudo-label ảnh hưởng gì tới mô hình? Nhìn confusion matrix và nhận xét.
4. Tại sao bài này **không phải một hệ thống cảnh báo AQI thật**?
5. Applied: trình bày bảng validation và lý do chọn tau; Foundation không bắt buộc.